Import necessary packages

In [3]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

Load datasets + data cleaning

In [9]:
%pip install swifter
import pandas as pd
import numpy as np
import ast
import swifter
from tqdm import tqdm
tqdm.pandas()

df_billing_history       = pd.read_csv("datasets/billing_history_all.csv")
df_billing_line_items    = pd.read_csv("datasets/billing_line_items_all.csv")
df_change_orders         = pd.read_csv("datasets/change_orders_all.csv")
df_contracts             = pd.read_csv("datasets/contracts_all.csv")
df_field_notes           = pd.read_csv("datasets/field_notes_all.csv")
df_labor_logs            = pd.read_csv("datasets/labor_logs_all.csv")
df_material_deliveries   = pd.read_csv("datasets/material_deliveries_all.csv")
df_rfis                  = pd.read_csv("datasets/rfis_all.csv")
df_sov                   = pd.read_csv("datasets/sov_all.csv")
df_sov_budget            = pd.read_csv("datasets/sov_budget_all.csv")

print(df_billing_history.head())

Note: you may need to restart the kernel to use updated packages.


C:\Users\yaoch\AppData\Local\Temp\ipykernel_21944\339465456.py:13: DtypeWarning: Columns (6,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_field_notes           = pd.read_csv("datasets/field_notes_all.csv")


     project_id  application_number  period_end  period_total  \
0  PRJ-2024-001                   2  2024-05-21       2944900   
1  PRJ-2024-001                   3  2024-06-20       3628300   
2  PRJ-2024-001                   4  2024-07-20       3206200   
3  PRJ-2024-001                   5  2024-08-19       2223700   
4  PRJ-2024-001                   6  2024-09-18       2092000   

   cumulative_billed  retention_held  net_payment_due status payment_date  \
0            2944900        294490.0        2650410.0   Paid   2024-06-30   
1            6573200        657320.0        5915880.0   Paid   2024-07-30   
2            9779400        977940.0        8801460.0   Paid   2024-08-27   
3           12003100       1200310.0       10802790.0   Paid   2024-09-16   
4           14095100       1409510.0       12685590.0   Paid   2024-10-27   

   line_item_count  
0                9  
1               12  
2               15  
3               15  
4               15  


In [ ]:
import duckdb

con = duckdb.connect()

contracts = con.read_csv("datasets/contracts_all.csv")
materials = con.read_csv("datasets/material_deliveries_all.csv")
laborlogs = con.read_csv("datasets/labor_logs_all.csv")

contracts_df = contracts.df()
contracts_df.head()

lldf = laborlogs.df()

con.execute("""
CREATE TABLE labor AS SELECT * FROM read_csv_auto('datasets/labor_logs_all.csv');
""")

con.execute("""
CREATE TABLE role_mapping AS SELECT * FROM read_csv_auto('role_mapping.csv');
""")

cleaned = con.execute("""
SELECT
    l.*,
    COALESCE(m.standard_role, l.role) AS standardized_role
FROM labor l
LEFT JOIN role_mapping m
ON l.role = m.original_role
""").fetchdf()
print(cleaned.columns)
cleaned["standardized_role"].value_counts()
